# 03 — Modelagem e banco de dados

Etapa 3 do enunciado: modelar as sete tabelas, discutir a normalização, criar o banco em **SQLite** com restrições de integridade, carregar os dados a partir do Python, verificar a integridade e responder às perguntas de negócio **usando apenas consultas SQL**.

O banco gerado fica em `banco/nice_rooms.db` e é a fonte dos notebooks 04 e do painel.

In [1]:
import sqlite3

import pandas as pd

from nice_rooms import ARQUIVO_BANCO, carregar

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)

## 1. Modelo Entidade-Relacionamento

Desenhei o modelo **antes** de criar as tabelas. O diagrama foi escrito em Mermaid (`banco/diagrama_er.mmd`) e exportado para `banco/diagrama_er.png` e `banco/diagrama_er.pdf`.

![Diagrama ER](../banco/diagrama_er.png)

**Entidades, chaves e cardinalidades**

| Relacionamento | Cardinalidade | Chave estrangeira |
|---|---|---|
| hotels → room_types | 1:N (um hotel tem 2 a 4 tipos de quarto) | `room_types.hotel_id` |
| hotels ↔ amenities | **N:N** (um hotel tem várias comodidades; uma comodidade aparece em vários hotéis) | resolvido pela tabela associativa `hotel_amenities (hotel_id, amenity_id)` |
| hotels → daily_market | 1:N (728 noites por hotel) | `daily_market.hotel_id` |
| hotels → weekly_performance | 1:N (104 semanas por hotel) | `weekly_performance.hotel_id` |
| weekly_performance → rate_decisions | **1:0..1** (cada semana tem no máximo uma decisão; a última semana não tem) | `rate_decisions (hotel_id, week_start)` |
| weekly_performance → reviews | 1:N (uma semana de um hotel recebe várias avaliações, ou nenhuma) | `reviews (hotel_id, week_start)` |
| review_scenarios → reviews | 1:N | `reviews.scenario_id` |
| room_types → reviews | 1:N | `reviews (hotel_id, room_type)` |

As tabelas `reviews` e `rate_decisions` também guardam `hotel_id` sozinho, com chave estrangeira para `hotels`. Ele faz parte das chaves compostas e permite filtrar por hotel sem junção.

**Chaves primárias:** naturais e simples onde o dataset já traz um identificador (`hotel_id`, `room_type_id`, `decision_id`, `review_id`, `scenario_id`). Compostas onde a linha é definida por "hotel + tempo" (`daily_market`: `hotel_id + date`; `weekly_performance`: `hotel_id + week_start`).

## 2. Normalização: o que remover e o que manter

O notebook 01 mostrou que várias colunas **repetem** informação de outras tabelas. A tabela abaixo resume a decisão para cada caso.

| Redundância | Forma normal violada | Decisão | Por quê |
|---|---|---|---|
| `amenities` como lista em um campo | **1FN** (valor não atômico) | **Remover** → `amenities` + `hotel_amenities` | Sem isso, "quantos hotéis têm restaurante?" exige processar texto. |
| `hotel_name`, `tier`, `comp_set` em `daily_market` e `weekly_performance`; `tier` em `room_types` | **3FN** (dependem de `hotel_id`, não da chave inteira) | **Remover** | São cópias: se um hotel mudasse de tier, seria preciso atualizar 10 mil linhas (anomalia de atualização). Recupero com junção ou com a *view* `vw_daily`. |
| `available_rooms` em `daily_market` | coluna duplicada | **Remover** | É apelido exato de `rooms_available`. |
| `review_count`, `lifetime_rating`, `share_rating_5`, `share_rating_1_2` em `hotels` | dado derivado | **Remover** → *view* `vw_hotel_reviews` | É um resumo que muda a cada nova avaliação. Calcular na hora garante que nunca fique desatualizado. |
| `demand_event`, `cancellation_rate` em `rate_decisions` | **3FN** (dependem da semana) | **Remover** | Idênticos aos de `weekly_performance` (confirmado abaixo), que é acessível pela chave estrangeira. |
| Indicadores calculados (`occupancy`, `adr`, `revpar`, `rgi`, `mars_rate`...) | dado derivado | **Manter** | São os valores "oficiais" publicados. Os índices de concorrentes (`comp_set_*`) **não podem** ser recalculados a partir das linhas, porque o README avisa que são ponderados. Recalcular tudo em cada consulta seria lento e sujeito a erro. |
| `weekday`, `month`, `year`, `season` em `daily_market` | derivados de `date` | **Manter** | Simplificam o `GROUP BY` (o SQLite tem poucas funções de data), e a estação não é uma função trivial da data. |
| `weekly_performance` inteira (agregação de `daily_market`) | dado derivado | **Manter** | Traz informação que não existe na diária (resumo de avaliações, `flow_through`) e é a "dona" das chaves usadas por `rate_decisions` e `reviews`. A semana já fechou, então o valor não muda mais. |

**Conclusão:** normalizo o que pode **ficar inconsistente** (cópias de atributos do hotel e resumos que mudam com o tempo) e mantenho a redundância que **ajuda a análise** sem risco de inconsistência (indicadores publicados, colunas de calendário, agregados de períodos fechados). Normalizar tudo ao extremo deixaria o banco mais "puro", mas tornaria as consultas mais lentas e mais difíceis de escrever.

Antes de remover as colunas de `rate_decisions`, confirmo que elas são mesmo cópias da tabela semanal:

In [2]:
decisoes = carregar("rate_decisions")
semanal = carregar("weekly_performance")
comparacao = decisoes.merge(semanal, on=["hotel_id", "week_start"], suffixes=("_dec", "_sem"))
print("demand_event idêntico:", (comparacao["demand_event_dec"] == comparacao["demand_event_sem"]).all())
print("cancellation_rate idêntico:",
      ((comparacao["cancellation_rate_dec"] - comparacao["cancellation_rate_sem"]).abs() < 1e-9).all())

demand_event idêntico: True
cancellation_rate idêntico: True


## 3. Criação do banco com restrições de integridade

Restrições usadas:

- `PRIMARY KEY` e `UNIQUE` para garantir a granularidade;
- `FOREIGN KEY` para as ligações (o SQLite só as aplica com `PRAGMA foreign_keys = ON`);
- `NOT NULL` nas colunas obrigatórias;
- `CHECK` para domínios: *tier* válido, ocupação entre 0 e 1, nota entre 1 e 5, ação de preço em {raise, hold, lower}, resultado em {better, in line, worse}.

Para não escrever à mão as 50 colunas numéricas de `daily_market` e `weekly_performance`, a função `colunas_sql` gera a definição a partir do DataFrame. As restrições importantes são escritas explicitamente.

In [3]:
ARQUIVO_BANCO.parent.mkdir(exist_ok=True)
ARQUIVO_BANCO.unlink(missing_ok=True)  # recria o banco do zero a cada execução
conexao = sqlite3.connect(ARQUIVO_BANCO)
conexao.execute("PRAGMA foreign_keys = ON")


def tipo_sql(serie: pd.Series) -> str:
    """Traduz o tipo do pandas para o tipo do SQLite."""
    if pd.api.types.is_bool_dtype(serie) or pd.api.types.is_integer_dtype(serie):
        return "INTEGER"
    if pd.api.types.is_float_dtype(serie):
        return "REAL"
    return "TEXT"


def colunas_sql(df: pd.DataFrame, ignorar: list[str]) -> str:
    """Gera 'coluna TIPO' para todas as colunas que não foram definidas à mão."""
    return ",\n    ".join(f"{c} {tipo_sql(df[c])}" for c in df.columns if c not in ignorar)


def preparar_datas(df: pd.DataFrame) -> pd.DataFrame:
    """Converte colunas de data para texto ISO (AAAA-MM-DD), o formato de data do SQLite."""
    df = df.copy()
    for c in df.select_dtypes("datetime").columns:
        df[c] = df[c].dt.strftime("%Y-%m-%d")
    return df

In [4]:
# --- Carrega e remove as colunas redundantes decididas na Seção 2 ---
hoteis = carregar("hotels")
tipos_quarto = carregar("room_types").drop(columns=["tier"])
diario = preparar_datas(carregar("daily_market").drop(columns=["hotel_name", "tier", "comp_set", "available_rooms"]))
semanal = preparar_datas(semanal.drop(columns=["hotel_name", "tier", "comp_set"]))
decisoes = preparar_datas(decisoes.drop(columns=["demand_event", "cancellation_rate"]))
avaliacoes = preparar_datas(carregar("reviews"))
cenarios = carregar("review_scenarios")

# Comodidades: 1FN (uma linha por hotel-comodidade) e tabela própria de comodidades (N:N)
hotel_comodidade = hoteis[["hotel_id", "amenities"]].explode("amenities").rename(columns={"amenities": "amenity"})
comodidades = (hotel_comodidade[["amenity"]].drop_duplicates().sort_values("amenity")
               .reset_index(drop=True).rename_axis("amenity_id").reset_index())
comodidades["amenity_id"] += 1
hotel_comodidade = hotel_comodidade.merge(comodidades, on="amenity")[["hotel_id", "amenity_id"]]

hoteis = hoteis.drop(columns=["amenities", "review_count", "lifetime_rating", "share_rating_5", "share_rating_1_2"])
print({"amenities": len(comodidades), "hotel_amenities": len(hotel_comodidade)})

{'amenities': 33, 'hotel_amenities': 57}


In [5]:
manuais_diario = ["hotel_id", "date", "occupancy", "rate_action", "rgi"]
manuais_semanal = ["hotel_id", "week_start", "occupancy", "mean_rating"]

DDL = f"""
CREATE TABLE hotels (
    hotel_id             TEXT PRIMARY KEY,
    name                 TEXT NOT NULL UNIQUE,
    brand                TEXT NOT NULL,
    tier                 TEXT NOT NULL CHECK (tier IN ('luxury','upscale','boutique','midscale','economy')),
    rooms                INTEGER NOT NULL CHECK (rooms > 0),
    opened_year          INTEGER,
    distance_to_water_km REAL CHECK (distance_to_water_km >= 0),
    neighbourhood        TEXT,
    comp_set             TEXT NOT NULL,
    positioning_index    REAL,
    base_rate            REAL CHECK (base_rate > 0),
    city                 TEXT
);

CREATE TABLE amenities (
    amenity_id INTEGER PRIMARY KEY,
    amenity    TEXT NOT NULL UNIQUE
);

CREATE TABLE hotel_amenities (
    hotel_id   TEXT    NOT NULL REFERENCES hotels(hotel_id),
    amenity_id INTEGER NOT NULL REFERENCES amenities(amenity_id),
    PRIMARY KEY (hotel_id, amenity_id)
);

CREATE TABLE room_types (
    room_type_id     TEXT PRIMARY KEY,
    hotel_id         TEXT NOT NULL REFERENCES hotels(hotel_id),
    room_type        TEXT NOT NULL,
    rooms            INTEGER NOT NULL CHECK (rooms > 0),
    max_occupancy    INTEGER,
    size_sqm         REAL,
    rate_premium_pct REAL,
    bar_rate         REAL,
    view             TEXT,
    UNIQUE (hotel_id, room_type)
);

CREATE TABLE review_scenarios (
    scenario_id        TEXT PRIMARY KEY,
    label              TEXT NOT NULL,
    share_of_reviews   REAL CHECK (share_of_reviews BETWEEN 0 AND 1),
    rating_band        TEXT,
    trigger            TEXT,
    expected_sentiment TEXT CHECK (expected_sentiment IN ('positive','mixed','negative')),
    recovery_action    TEXT
);

CREATE TABLE daily_market (
    hotel_id    TEXT NOT NULL REFERENCES hotels(hotel_id),
    date        TEXT NOT NULL,
    occupancy   REAL NOT NULL CHECK (occupancy BETWEEN 0 AND 1),
    rgi         REAL CHECK (rgi > 0),
    rate_action TEXT NOT NULL CHECK (rate_action IN ('raise','hold','lower')),
    {colunas_sql(diario, manuais_diario)},
    PRIMARY KEY (hotel_id, date)
);

CREATE TABLE weekly_performance (
    hotel_id    TEXT NOT NULL REFERENCES hotels(hotel_id),
    week_start  TEXT NOT NULL,
    occupancy   REAL NOT NULL CHECK (occupancy BETWEEN 0 AND 1),
    mean_rating REAL CHECK (mean_rating BETWEEN 1 AND 5),  -- nulo quando não houve avaliação
    {colunas_sql(semanal, manuais_semanal)},
    PRIMARY KEY (hotel_id, week_start)
);

CREATE TABLE rate_decisions (
    decision_id          TEXT PRIMARY KEY,
    hotel_id             TEXT NOT NULL REFERENCES hotels(hotel_id),
    week_start           TEXT NOT NULL,
    action               TEXT NOT NULL CHECK (action IN ('raise','hold','lower')),
    outcome              TEXT NOT NULL CHECK (outcome IN ('better','in line','worse')),
    {colunas_sql(decisoes, ["decision_id", "hotel_id", "week_start", "action", "outcome"])},
    UNIQUE (hotel_id, week_start),
    FOREIGN KEY (hotel_id, week_start) REFERENCES weekly_performance(hotel_id, week_start)
);

CREATE TABLE reviews (
    review_id      TEXT PRIMARY KEY,
    hotel_id       TEXT NOT NULL REFERENCES hotels(hotel_id),
    week_start     TEXT NOT NULL,
    scenario_id    TEXT NOT NULL REFERENCES review_scenarios(scenario_id),
    room_type      TEXT NOT NULL,
    rating         INTEGER NOT NULL CHECK (rating BETWEEN 1 AND 5),
    response_hours REAL CHECK (response_hours >= 0),  -- nulo quando o hotel não respondeu
    {colunas_sql(avaliacoes, ["review_id", "hotel_id", "week_start", "scenario_id", "room_type", "rating", "response_hours"])},
    FOREIGN KEY (hotel_id, week_start) REFERENCES weekly_performance(hotel_id, week_start),
    FOREIGN KEY (hotel_id, room_type)  REFERENCES room_types(hotel_id, room_type)
);

CREATE INDEX idx_reviews_hotel ON reviews(hotel_id);
CREATE INDEX idx_reviews_scenario ON reviews(scenario_id);
"""
conexao.executescript(DDL)
print(pd.read_sql("SELECT name, type FROM sqlite_master WHERE type IN ('table','index') ORDER BY type DESC, name", conexao))

                                     name   type
0                               amenities  table
1                            daily_market  table
2                         hotel_amenities  table
3                                  hotels  table
4                          rate_decisions  table
5                        review_scenarios  table
6                                 reviews  table
7                              room_types  table
8                      weekly_performance  table
9                       idx_reviews_hotel  index
10                   idx_reviews_scenario  index
11           sqlite_autoindex_amenities_1  index
12        sqlite_autoindex_daily_market_1  index
13     sqlite_autoindex_hotel_amenities_1  index
14              sqlite_autoindex_hotels_1  index
15              sqlite_autoindex_hotels_2  index
16      sqlite_autoindex_rate_decisions_1  index
17      sqlite_autoindex_rate_decisions_2  index
18    sqlite_autoindex_review_scenarios_1  index
19             sqlit

### 3.1 Carga dos dados

A ordem importa: primeiro as tabelas "mãe" (`hotels`, `amenities`, `review_scenarios`), depois as que dependem delas. Com `foreign_keys = ON`, uma linha que apontasse para um hotel inexistente seria **recusada** na carga.

In [6]:
ORDEM_CARGA = [
    ("hotels", hoteis), ("amenities", comodidades), ("hotel_amenities", hotel_comodidade),
    ("room_types", tipos_quarto), ("review_scenarios", cenarios), ("daily_market", diario),
    ("weekly_performance", semanal), ("rate_decisions", decisoes), ("reviews", avaliacoes),
]
with conexao:
    for nome, df in ORDEM_CARGA:
        df.to_sql(nome, conexao, if_exists="append", index=False)

for nome, _ in ORDEM_CARGA:
    total = conexao.execute(f"SELECT COUNT(*) FROM {nome}").fetchone()[0]
    print(f"{nome:<20} {total:>6} linhas")

hotels                   14 linhas
amenities                33 linhas
hotel_amenities          57 linhas
room_types               42 linhas
review_scenarios         11 linhas
daily_market          10192 linhas
weekly_performance     1456 linhas
rate_decisions         1442 linhas
reviews               12077 linhas


**Teste das restrições.** Tento inserir uma avaliação de um hotel que não existe e uma nota 6. As duas devem ser recusadas.

In [7]:
testes = {
    "hotel inexistente": "INSERT INTO reviews (review_id, hotel_id, week_start, scenario_id, room_type, rating) "
                         "VALUES ('TESTE1', 'H99', '2027-01-04', 'noise', 'Standard King', 2)",
    "nota 6": "INSERT INTO reviews (review_id, hotel_id, week_start, scenario_id, room_type, rating) "
              "VALUES ('TESTE2', 'H01', '2027-01-04', 'noise', 'Standard King', 6)",
}
for descricao, comando in testes.items():
    try:
        conexao.execute(comando)
        print(f"{descricao}: ACEITO (problema!)")
    except sqlite3.IntegrityError as erro:
        print(f"{descricao}: recusado -> {erro}")
conexao.rollback()

hotel inexistente: recusado -> FOREIGN KEY constraint failed
nota 6: recusado -> CHECK constraint failed: rating BETWEEN 1 AND 5


### 3.2 Views: a redundância de volta, sem risco

As colunas removidas continuam disponíveis por **views**, que são consultas salvas e calculadas na hora:

- `vw_daily`: a tabela diária com nome, tier e receita total do hotel;
- `vw_hotel_reviews`: o resumo de avaliações por hotel que antes ficava em `hotels`.

In [8]:
conexao.executescript("""
CREATE VIEW vw_daily AS
SELECT h.name AS hotel_name, h.tier, d.*, d.room_revenue + d.ancillary_revenue AS total_revenue
FROM daily_market d JOIN hotels h ON h.hotel_id = d.hotel_id;

CREATE VIEW vw_hotel_reviews AS
SELECT hotel_id,
       COUNT(*)                                          AS review_count,
       ROUND(AVG(rating), 3)                             AS lifetime_rating,
       ROUND(AVG(CASE WHEN rating = 5 THEN 1.0 ELSE 0 END), 4)  AS share_rating_5,
       ROUND(AVG(CASE WHEN rating <= 2 THEN 1.0 ELSE 0 END), 4) AS share_rating_1_2
FROM reviews GROUP BY hotel_id;
""")
conexao.commit()


def consulta(sql: str) -> pd.DataFrame:
    """Executa uma consulta SQL no banco e devolve um DataFrame."""
    return pd.read_sql(sql, conexao)


# A view reproduz exatamente os valores originais do cadastro?
original = carregar("hotels").set_index("hotel_id")[["review_count", "lifetime_rating", "share_rating_1_2"]]
view = consulta("SELECT * FROM vw_hotel_reviews").set_index("hotel_id")
print("review_count igual:", (original["review_count"] == view["review_count"]).all())
print("lifetime_rating igual:", ((original["lifetime_rating"] - view["lifetime_rating"]).abs() < 0.001).all())
print("share_rating_1_2 igual:", ((original["share_rating_1_2"] - view["share_rating_1_2"]).abs() < 0.0001).all())

review_count igual: True
lifetime_rating igual: True
share_rating_1_2 igual: True


## 4. Verificação de integridade

Além das chaves estrangeiras (que já barrariam a carga), confirmo com consultas que **nenhuma avaliação, decisão ou registro diário aponta para um hotel inexistente**. O padrão é `LEFT JOIN ... WHERE pai IS NULL`: ele lista as linhas da tabela filha sem par na tabela mãe.

In [9]:
consulta("""
SELECT 'reviews' AS tabela, COUNT(*) AS orfaos
FROM reviews r LEFT JOIN hotels h ON h.hotel_id = r.hotel_id WHERE h.hotel_id IS NULL
UNION ALL
SELECT 'rate_decisions', COUNT(*)
FROM rate_decisions d LEFT JOIN hotels h ON h.hotel_id = d.hotel_id WHERE h.hotel_id IS NULL
UNION ALL
SELECT 'daily_market', COUNT(*)
FROM daily_market d LEFT JOIN hotels h ON h.hotel_id = d.hotel_id WHERE h.hotel_id IS NULL
UNION ALL
SELECT 'weekly_performance', COUNT(*)
FROM weekly_performance w LEFT JOIN hotels h ON h.hotel_id = w.hotel_id WHERE h.hotel_id IS NULL
UNION ALL
SELECT 'reviews -> semana', COUNT(*)
FROM reviews r LEFT JOIN weekly_performance w ON w.hotel_id = r.hotel_id AND w.week_start = r.week_start
WHERE w.hotel_id IS NULL
UNION ALL
SELECT 'reviews -> cenário', COUNT(*)
FROM reviews r LEFT JOIN review_scenarios s ON s.scenario_id = r.scenario_id WHERE s.scenario_id IS NULL
""")

,tabela,orfaos
0,reviews,0
1,rate_decisions,0
2,daily_market,0
3,weekly_performance,0
4,reviews -> semana,0
5,reviews -> cenário,0


In [10]:
# O SQLite também tem um verificador próprio: devolve uma linha para cada violação de chave estrangeira
violacoes = consulta("PRAGMA foreign_key_check")
print("Violações encontradas pelo PRAGMA foreign_key_check:", len(violacoes))

# Granularidade esperada: 728 noites, 104 semanas e 103 decisões por hotel
consulta("""
SELECT h.hotel_id,
       (SELECT COUNT(*) FROM daily_market d WHERE d.hotel_id = h.hotel_id)       AS noites,
       (SELECT COUNT(*) FROM weekly_performance w WHERE w.hotel_id = h.hotel_id) AS semanas,
       (SELECT COUNT(*) FROM rate_decisions r WHERE r.hotel_id = h.hotel_id)     AS decisoes
FROM hotels h
""").drop_duplicates(subset=["noites", "semanas", "decisoes"])

Violações encontradas pelo PRAGMA foreign_key_check: 0


,hotel_id,noites,semanas,decisoes
0,H01,728,104,103


**Integridade confirmada:** zero órfãos em todas as ligações, nenhuma violação de chave estrangeira e o mesmo número de noites, semanas e decisões para todos os hotéis.

---
## 5. Perguntas de negócio (somente SQL)

### 5.1 Ranking dos hotéis por receita total em cada ano

Receita total = receita de quartos + receita extra (alimentação, bebidas etc.), somada noite a noite. Uso a função de janela `RANK() OVER (PARTITION BY ano ...)` para numerar o ranking **dentro de cada ano**.

In [11]:
ranking = consulta("""
SELECT d.year AS ano,
       RANK() OVER (PARTITION BY d.year ORDER BY SUM(d.room_revenue + d.ancillary_revenue) DESC) AS posicao,
       h.name AS hotel, h.tier,
       ROUND(SUM(d.room_revenue + d.ancillary_revenue) / 1e6, 2) AS receita_mi_usd
FROM daily_market d JOIN hotels h ON h.hotel_id = d.hotel_id
GROUP BY d.year, h.hotel_id
ORDER BY ano, posicao
""")
ranking.pivot(index="posicao", columns="ano", values=["hotel", "receita_mi_usd"])

hotel                      receita_mi_usd       
ano                     2027                 2028           2027   2028
posicao                                                                
1         Marisol Bay Resort   Marisol Bay Resort          26.12  26.38
2            Seabright Grand      Seabright Grand          16.14  16.22
3         Alder & Quay Hotel   Alder & Quay Hotel           12.3  12.48
4         Harborlight Suites   Harborlight Suites          10.99  11.12
5        The Copper Meridian  The Copper Meridian          10.61  10.67
6         Kestrel Park Hotel   Kestrel Park Hotel           8.24    8.3
7           Willowmere Lodge     Willowmere Lodge           6.52   6.55
8         Cedar & Salt Hotel   Cedar & Salt Hotel           6.24   6.32
9         Paloma Court Hotel   Paloma Court Hotel           5.76   5.84
10         The Lantern House    The Lantern House           4.61   4.64
11             Northgate Inn        Northgate Inn           3.09   3.11
12              The Foxglove         The Foxglove           2.54   2.56
13            The Wren Rooms       The Wren Rooms           2.44   2.46
14       Ridgeline Motor Inn  Ridgeline Motor Inn           1.87   1.88

**Interpretação.** O ranking é **idêntico nos dois anos**. O Marisol Bay Resort (luxury, 212 quartos) lidera com ~26 mi USD por ano, seguido do Seabright Grand (upscale, o maior hotel, 240 quartos) e do Alder & Quay. O Ridgeline Motor Inn (economy) fecha a lista. A receita total depende muito do **tamanho** do hotel: para comparar desempenho é preciso olhar indicadores por quarto (RevPAR) e contra os concorrentes (RGI), como nas próximas consultas e no painel.

### 5.2 Ocupação e diária médias por tier e estação

Calculo a ocupação como **total de quartos vendidos ÷ total de quartos disponíveis** e a ADR como **receita de quartos ÷ quartos vendidos**. Tirar a média das porcentagens diárias daria o mesmo peso a um hotel de 38 e a um de 240 quartos. A razão das somas é a definição do setor para um grupo de hotéis.

In [12]:
tier_estacao = consulta("""
SELECT h.tier, d.season AS estacao,
       ROUND(100.0 * SUM(d.rooms_sold) / SUM(d.rooms_available), 1) AS ocupacao_pct,
       ROUND(SUM(d.room_revenue) / SUM(d.rooms_sold), 2)            AS adr_usd
FROM daily_market d JOIN hotels h ON h.hotel_id = d.hotel_id
GROUP BY h.tier, d.season
ORDER BY CASE h.tier WHEN 'luxury' THEN 1 WHEN 'upscale' THEN 2 WHEN 'boutique' THEN 3
                     WHEN 'midscale' THEN 4 ELSE 5 END,
         CASE d.season WHEN 'winter' THEN 1 WHEN 'spring' THEN 2 WHEN 'summer' THEN 3 ELSE 4 END
""")
tier_estacao.pivot(index="tier", columns="estacao", values=["ocupacao_pct", "adr_usd"]).reindex(
    ["luxury", "upscale", "boutique", "midscale", "economy"])[
    [(m, e) for m in ["ocupacao_pct", "adr_usd"] for e in ["winter", "spring", "summer", "autumn"]]]

ocupacao_pct                      adr_usd                        
estacao        winter spring summer autumn  winter  spring  summer  autumn
tier                                                                      
luxury           63.1   71.6   83.3   71.6  322.20  366.02  454.65  369.96
upscale          63.7   72.4   84.1   72.2  192.56  218.64  271.79  220.75
boutique         66.1   74.9   86.7   74.9  185.38  210.22  261.52  212.83
midscale         59.5   67.5   78.6   67.4  131.56  148.86  184.73  150.51
economy          54.0   61.5   71.8   61.2   80.10   90.06  111.41   90.93

**Interpretação.** Todos os *tiers* seguem o mesmo ritmo: o **verão** é a melhor estação (≈ +12 pp de ocupação e ≈ +25% de diária em relação à primavera e ao outono), e o **inverno** é a pior. Primavera e outono são quase iguais. O nível de preço muda muito entre *tiers* (o luxo cobra 4 a 5 vezes o econômico), por isso a comparação de preços só faz sentido dentro de cada linha.

### 5.3 Hotéis com mais de 30% de avaliações de 1 ou 2 estrelas

In [13]:
consulta("""
SELECT h.name AS hotel, h.tier,
       COUNT(*) AS avaliacoes,
       SUM(CASE WHEN r.rating <= 2 THEN 1 ELSE 0 END) AS notas_1_2,
       ROUND(100.0 * SUM(CASE WHEN r.rating <= 2 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_1_2
FROM reviews r JOIN hotels h ON h.hotel_id = r.hotel_id
GROUP BY h.hotel_id
HAVING 1.0 * SUM(CASE WHEN r.rating <= 2 THEN 1 ELSE 0 END) / COUNT(*) > 0.30
ORDER BY pct_1_2 DESC
""")

,hotel,tier,avaliacoes,notas_1_2,pct_1_2
0,Northgate Inn,economy,718,281,39.1
1,Ridgeline Motor Inn,economy,506,184,36.4
2,The Wren Rooms,boutique,289,103,35.6
3,Kestrel Park Hotel,midscale,1169,411,35.2
4,Willowmere Lodge,upscale,679,221,32.5
5,Cedar & Salt Hotel,midscale,963,304,31.6
6,Paloma Court Hotel,midscale,829,251,30.3


**Interpretação.** **Sete hotéis** passam de 30% de avaliações ruins, e os dois econômicos estão no topo (Northgate Inn ~39%, Ridgeline Motor Inn ~36%). Também aparecem os três midscale, um boutique (The Wren Rooms) e um upscale (Willowmere Lodge). Nenhum hotel de luxo está na lista. Isso combina com o README: hotéis econômicos e midscale recebem mais casos de conservação, barulho, cobrança e limpeza.

### 5.4 Cenários de avaliação: volume, nota média e ação recomendada

Esta consulta **junta** `reviews` (as avaliações) com `review_scenarios` (o motivo e a ação de recuperação).

In [14]:
consulta("""
SELECT s.scenario_id, s.label AS cenario,
       COUNT(r.review_id) AS avaliacoes,
       ROUND(AVG(r.rating), 2) AS nota_media,
       s.recovery_action AS acao_recomendada
FROM review_scenarios s JOIN reviews r ON r.scenario_id = s.scenario_id
GROUP BY s.scenario_id
ORDER BY avaliacoes DESC
""")

,scenario_id,cenario,avaliacoes,nota_media,acao_recomendada
0,delight,Stay exceeded expectations,1840,4.84,thank the guest and record the staff member
1,room_condition,Room condition,1401,2.48,block the room and raise a work order
2,noise,Noise and sleep,1250,2.42,"offer a move and a sleep kit, note the room"
3,location,Location and access,1219,4.58,none needed
4,service_recovery,Service recovery,1206,4.33,log the recovery and brief the shift
5,billing,Billing and charges,1027,1.63,audit the folio and refund where due
6,breakfast,Breakfast and food,1023,3.73,invite the guest back and review the buffet line
7,checkin_wait,Check-in and queue,1006,2.81,add a queue check at peak arrival
8,cleanliness,Cleanliness,868,1.69,"reclean, inspect and coach the section"
9,wifi,Wifi and connectivity,765,2.74,send the access point log to IT


**Interpretação.** O cenário mais frequente é positivo ("Stay exceeded expectations", ~1.840 avaliações, nota 4,84). Logo depois vêm os dois problemas mais comuns: **conservação do quarto** e **barulho** (~1.400 e ~1.250 avaliações, notas ≈ 2,4–2,5). **Cobrança** e **limpeza** são menos frequentes, mas têm as piores notas (≈ 1,6–1,7). Para a diretoria, as duas ações de maior alcance são **bloquear o quarto e abrir ordem de serviço** e **oferecer troca de quarto e kit de sono**.

### 5.5 Decisões de preço: proporção de semanas seguintes "melhores"

In [15]:
decisoes_resultado = consulta("""
SELECT action AS acao,
       COUNT(*) AS decisoes,
       ROUND(100.0 * SUM(CASE WHEN outcome = 'better'  THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_melhor,
       ROUND(100.0 * SUM(CASE WHEN outcome = 'in line' THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_igual,
       ROUND(100.0 * SUM(CASE WHEN outcome = 'worse'   THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_pior,
       ROUND(AVG(realised_uplift_pct), 2) AS variacao_media_revpar_pct
FROM rate_decisions
GROUP BY action
ORDER BY CASE action WHEN 'raise' THEN 1 WHEN 'hold' THEN 2 ELSE 3 END
""")
decisoes_resultado

,acao,decisoes,pct_melhor,pct_igual,pct_pior,variacao_media_revpar_pct
0,raise,634,38.3,30.8,30.9,0.45
1,hold,619,37.6,31.8,30.5,0.68
2,lower,189,42.9,26.5,30.7,0.60


**Interpretação.** Os três tipos de decisão têm **proporções muito parecidas** de semanas seguintes "melhores": cerca de 38% para aumentar, 38% para manter e 43% para reduzir. Em todos os casos, cerca de um terço das semanas foi pior.

Isso **não prova** que as decisões funcionam nem que não funcionam. O resultado é **observacional**: compara a semana seguinte com a semana da decisão, e a semana seguinte muda por sazonalidade e eventos, não só pelo preço. O README avisa que o contrafactual ("o que teria acontecido sem a decisão") não foi simulado. O que se pode dizer: **não há sinal de que uma das ações seja sistematicamente melhor**, e o número de reduções (189) é pequeno.

### 5.6 Respostas do hotel por faixa de nota e tempo de resposta

`response_hours` é nulo quando não houve resposta. `AVG` no SQL **ignora nulos**, então a média de horas já é calculada só sobre as avaliações respondidas.

In [16]:
consulta("""
SELECT CASE WHEN rating <= 2 THEN '1-2 estrelas' WHEN rating = 3 THEN '3 estrelas'
            ELSE '4-5 estrelas' END AS faixa,
       COUNT(*) AS avaliacoes,
       SUM(CASE WHEN response_hours IS NOT NULL THEN 1 ELSE 0 END) AS respondidas,
       ROUND(100.0 * SUM(CASE WHEN response_hours IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_respondidas,
       ROUND(AVG(response_hours), 1) AS horas_media_resposta
FROM reviews
GROUP BY faixa
ORDER BY faixa
""")

,faixa,avaliacoes,respondidas,pct_respondidas,horas_media_resposta
0,1-2 estrelas,3526,3526,100.0,21.2
1,3 estrelas,3212,3212,100.0,21.1
2,4-5 estrelas,5339,1570,29.4,20.9


In [17]:
consulta("""
SELECT ROUND(100.0 * COUNT(response_hours) / COUNT(*), 1) AS pct_respondidas_geral,
       ROUND(AVG(response_hours), 1) AS horas_media_geral
FROM reviews
""")

,pct_respondidas_geral,horas_media_geral
0,68.8,21.1


**Interpretação.** Os hotéis respondem **100% das avaliações de 1 a 3 estrelas** e só **~29% das de 4 e 5 estrelas**. No total, ~69% das avaliações recebem resposta, em média **~21 horas** depois. A política parece ser "responder sempre às críticas". Agradecer a quem elogiou é uma oportunidade pouco explorada.

### 5.7 Pergunta própria: quais hotéis perdem participação justamente nas noites de evento?

Nas noites de evento a demanda da cidade sobe. Um hotel com **RGI médio abaixo de 100 nessas noites** está ganhando menos receita por quarto do que os concorrentes diretos justamente quando há mais dinheiro na mesa. A consulta junta `daily_market` e `hotels`, filtra as noites de evento (`WHERE`), agrupa por hotel (`GROUP BY`) e filtra os grupos (`HAVING`). Para comparação, mostro também o RGI nas noites comuns.

In [18]:
consulta("""
SELECT h.name AS hotel, h.tier,
       COUNT(*) AS noites_evento,
       ROUND(AVG(d.rgi), 1) AS rgi_evento,
       (SELECT ROUND(AVG(d2.rgi), 1) FROM daily_market d2
         WHERE d2.hotel_id = h.hotel_id AND d2.demand_event = '') AS rgi_noites_comuns,
       ROUND(AVG(d.rate_gap_pct), 1) AS desvio_mars_pct
FROM daily_market d JOIN hotels h ON h.hotel_id = d.hotel_id
WHERE d.demand_event <> ''
GROUP BY h.hotel_id
HAVING AVG(d.rgi) < 100
ORDER BY rgi_evento
""")

,hotel,tier,noites_evento,rgi_evento,rgi_noites_comuns,desvio_mars_pct
0,The Foxglove,boutique,47,76.4,76.0,-18.7
1,Ridgeline Motor Inn,economy,47,85.6,85.3,-5.9
2,Willowmere Lodge,upscale,47,87.1,87.1,-8.8
3,The Wren Rooms,boutique,47,88.7,89.4,-3.3
4,The Copper Meridian,luxury,47,89.1,89.4,-4.1
5,Cedar & Salt Hotel,midscale,47,91.4,91.2,-4.4
6,Seabright Grand,upscale,47,93.6,93.3,-1.8
7,Paloma Court Hotel,midscale,47,99.4,99.6,-0.5


**Interpretação.** **Oito hotéis** ficam abaixo da paridade (RGI < 100) nas noites de evento. O caso mais grave é **The Foxglove** (boutique, RGI ~76), que também cobra ~19% abaixo da diária MARS. O RGI dos hotéis praticamente **não muda** entre noites de evento e noites comuns: quem perde participação, perde sempre, e o evento não corrige. Isso aponta para um problema estrutural de **preço** (cobram abaixo da MARS) e não de demanda. É o caso típico de candidato a aumento de diária.

---
## 6. (Bônus) Avaliações em um banco de documentos — MongoDB via *mongita*

No modelo relacional, uma avaliação está espalhada em quatro tabelas (`reviews`, `hotels`, `review_scenarios`, `room_types`). Em um banco de documentos, cada avaliação vira **um documento** que já carrega o que é preciso para exibi-la: o hotel, o cenário, as notas parciais agrupadas e a resposta do hotel, que existe ou não (sem colunas nulas).

In [19]:
from mongita import MongitaClientMemory

cliente = MongitaClientMemory()
colecao = cliente.nice_rooms.reviews
colecao.delete_many({})

docs = consulta("""
SELECT r.*, h.name AS hotel_name, h.tier, s.label AS scenario_label, s.recovery_action
FROM reviews r JOIN hotels h ON h.hotel_id = r.hotel_id JOIN review_scenarios s ON s.scenario_id = r.scenario_id
""")


def para_documento(linha) -> dict:
    """Monta um documento aninhado a partir de uma linha da junção."""
    documento = {
        "_id": linha.review_id,
        "hotel": {"id": linha.hotel_id, "name": linha.hotel_name, "tier": linha.tier},
        "stay": {"date": linha.stay_date, "nights": int(linha.nights), "room_type": linha.room_type,
                 "segment": linha.segment},
        "scenario": {"id": linha.scenario_id, "label": linha.scenario_label, "recovery": linha.recovery_action},
        "rating": int(linha.rating),
        "scores": {"cleanliness": int(linha.cleanliness), "comfort": int(linha.comfort),
                   "service": int(linha.service_score), "location": int(linha.location_score),
                   "value": int(linha.value_score), "sleep": int(linha.sleep_quality)},
        "title": linha.title, "text": linha.text, "channel": linha.channel,
        "sentiment": linha.sentiment,
    }
    if linha.response_text:  # só existe quando o hotel respondeu
        documento["response"] = {"text": linha.response_text, "hours": float(linha.response_hours)}
    return documento


colecao.insert_many([para_documento(l) for l in docs.itertuples()])
print("documentos:", colecao.count_documents({}))
exemplo = colecao.find_one({"hotel.id": "H08", "rating": 1})
exemplo

documentos: 12077


{'_id': 'R000027',
 'hotel': {'id': 'H08', 'name': 'Northgate Inn', 'tier': 'economy'},
 'stay': {'date': '2027-01-04',
  'nights': 1,
  'room_type': 'Twin Room',
  'segment': 'corporate'},
 'scenario': {'id': 'cleanliness',
  'label': 'Cleanliness',
  'recovery': 'reclean, inspect and coach the section'},
 'rating': 1,
 'scores': {'cleanliness': 1,
  'comfort': 1,
  'service': 1,
  'location': 3,
  'value': 1,
  'sleep': 1},
 'title': 'Needs attention',
 'text': 'The Twin Room had not been vacuumed and the bathroom smelled of damp. We asked twice for a reclean. We expected far more for the money. The car park was full despite the booking.',
 'channel': 'social',
 'sentiment': 'negative',
 'response': {'text': 'That is a fair complaint and we have acted on it. We hope you will give us another chance.',
  'hours': 3.0}}

In [20]:
# Consultas no estilo MongoDB: filtros sobre campos aninhados, sem junções
ruins_economy = colecao.count_documents({"hotel.tier": "economy", "rating": {"$lte": 2}})
total_economy = colecao.count_documents({"hotel.tier": "economy"})
print(f"Economy: {ruins_economy} de {total_economy} avaliações com 1-2 estrelas "
      f"({100 * ruins_economy / total_economy:.1f}%)")

# O mongita não implementa o operador $exists do MongoDB; filtro as 5 estrelas e testo o campo em Python
sem_resposta_5 = sum(1 for doc in colecao.find({"rating": 5}) if "response" not in doc)
print("Avaliações 5 estrelas sem resposta:", sem_resposta_5)

for doc in colecao.find({"hotel.id": "H11", "scenario.id": "billing"}).limit(2):
    print("-", doc["title"], "|", doc["scenario"]["recovery"])

Economy: 465 de 1224 avaliações com 1-2 estrelas (38.0%)
Avaliações 5 estrelas sem resposta: 2130
- Poor stay | audit the folio and refund where due
- Not worth the rate | audit the folio and refund where due


**Quando o banco de documentos seria mais vantajoso?**

- **Leitura de uma avaliação inteira** (por exemplo, a tela de um aplicativo que mostra a avaliação com hotel, cenário e resposta): um único documento, sem quatro junções.
- **Estrutura variável:** respostas que existem ou não, e no futuro fotos, várias respostas ou notas parciais novas. O documento acomoda isso sem `ALTER TABLE` nem colunas cheias de nulos.
- **Grande volume de escrita** de textos, como um fluxo contínuo de avaliações vindo de vários canais.

**Quando o relacional é melhor (o caso deste trabalho):** análises que **cruzam** avaliações com desempenho financeiro e decisões de preço, agregações por hotel/semana/tier e garantia de integridade (chaves estrangeiras, `CHECK`). No documento, o nome e o tier do hotel ficam **duplicados** em 12 mil documentos. Se o hotel mudasse de nome, todos teriam de ser atualizados, que é exatamente a anomalia que a normalização evita.

In [21]:
conexao.close()
print("Banco salvo em:", ARQUIVO_BANCO.resolve(), f"({ARQUIVO_BANCO.stat().st_size / 1e6:.1f} MB)")

Banco salvo em: /Users/walterpaixaocortes/projects/demo-dash/modelo/banco/nice_rooms.db (11.0 MB)
